# Student Placement & Salary Prediction

## Step 1: Setup (folders, CSV check, libraries)

In [11]:
import os, sys, subprocess, importlib.util

os.makedirs("pages", exist_ok=True)
os.makedirs("saved_models", exist_ok=True)

assert os.path.exists("studentplacement.csv"), "studentplacement.csv is notebook wale folder me rakho"

need = {"streamlit": "streamlit", "sklearn": "scikit-learn", "pandas": "pandas",
        "numpy": "numpy", "matplotlib": "matplotlib"}
missing = [pip_name for mod, pip_name in need.items() if importlib.util.find_spec(mod) is None]
if missing:
    subprocess.check_call([sys.executable, "-m", "pip", "install", *missing])
print("Folder:", os.getcwd())
print("Missing libraries installed:", missing if missing else "none, sab pehle se hain")

Folder: C:\Users\Admin\clude\student_placement_project
Missing libraries installed: none, sab pehle se hain


## Step 2: `utils.py` (data prep, 5 models training, pkl saving, dashboard style)

In [12]:
%%writefile utils.py
import streamlit as st
import pandas as pd
import numpy as np
import os
import pickle

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor, ExtraTreesRegressor, GradientBoostingRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score



CSV_FILE = "studentplacement.csv"
TARGET = "salary_package_lpa"

MODEL_FOLDER = "saved_models"



SELECTED_FEATURES = [
    "cgpa",
    "branch",
    "college_tier",
    "internships_count",
    "projects_count",
    "certifications_count",
    "coding_skill_score",
    "aptitude_score",
    "communication_skill_score",
    "mock_interview_score",
    "attendance_percentage",
    "backlogs"
]



@st.cache_data
def load_raw_data():

    return pd.read_csv(CSV_FILE)



@st.cache_data
def prepare_data(raw):

    df = raw.copy()

    df.fillna(
        df.mean(numeric_only=True),
        inplace=True
    )

    df = df.drop_duplicates().reset_index(drop=True)

    encoders = {}

    for col in df.select_dtypes(include="object").columns:

        le = LabelEncoder()

        df[col] = le.fit_transform(
            df[col].astype(str)
        )

        encoders[col] = le

    return df, encoders



def save_model(model, file_name):

    # Create folder if it does not exist
    os.makedirs(MODEL_FOLDER, exist_ok=True)

    file_path = os.path.join(
        MODEL_FOLDER,
        file_name
    )

    with open(file_path, "wb") as file:

        pickle.dump(
            model,
            file
        )



def save_encoders(encoders):

    os.makedirs(
        MODEL_FOLDER,
        exist_ok=True
    )

    file_path = os.path.join(
        MODEL_FOLDER,
        "encoders.pkl"
    )

    with open(file_path, "wb") as file:

        pickle.dump(
            encoders,
            file
        )



@st.cache_resource
def train_all_models(
    df,
    _encoders,
    model_version="v3"
):


    placed_value = _encoders["placement_status"].transform(
        ["Placed"]
    )[0]



    salary_df = df[
        df["placement_status"] == placed_value
    ].copy()



    X = salary_df[SELECTED_FEATURES].copy()

    y = salary_df[TARGET].copy()



    X_train, X_test, y_train, y_test = train_test_split(
        X,
        y,
        test_size=0.20,
        random_state=42
    )


    lr = LinearRegression()

    dt = DecisionTreeRegressor(
        max_depth=15,
        min_samples_split=10,
        min_samples_leaf=3,
        random_state=42
    )


    rf = RandomForestRegressor(
        n_estimators=500,
        max_depth=25,
        min_samples_split=4,
        min_samples_leaf=2,
        max_features=0.8,
        bootstrap=True,
        random_state=42,
        n_jobs=-1
    )


    extra = ExtraTreesRegressor(
        n_estimators=400,
        max_depth=None,
        min_samples_split=3,
        min_samples_leaf=1,
        max_features=0.9,
        random_state=42,
        n_jobs=-1
    )



    gb = GradientBoostingRegressor(
        n_estimators=300,
        learning_rate=0.05,
        max_depth=4,
        min_samples_split=5,
        min_samples_leaf=2,
        loss="huber",
        random_state=42
    )



    lr.fit(
        X_train,
        y_train
    )

    dt.fit(
        X_train,
        y_train
    )

    rf.fit(
        X_train,
        y_train
    )

    extra.fit(
        X_train,
        y_train
    )

    gb.fit(
        X_train,
        y_train
    )


    pred_lr = lr.predict(X_test)

    pred_dt = dt.predict(X_test)

    pred_rf = rf.predict(X_test)

    pred_extra = extra.predict(X_test)

    pred_gb = gb.predict(X_test)



    models = {

        "Linear Regression": lr,

        "Decision Tree": dt,

        "Random Forest": rf,

        "Extra Trees": extra,

        "Gradient Boosting": gb
    }



    save_model(
        lr,
        "linear_regression.pkl"
    )

    save_model(
        dt,
        "decision_tree.pkl"
    )

    save_model(
        rf,
        "random_forest.pkl"
    )

    save_model(
        extra,
        "extra_trees.pkl"
    )

    save_model(
        gb,
        "gradient_boosting.pkl"
    )


    # Save encoders also
    save_encoders(_encoders)



    predictions = {

        "Linear Regression": pred_lr,

        "Decision Tree": pred_dt,

        "Random Forest": pred_rf,

        "Extra Trees": pred_extra,

        "Gradient Boosting": pred_gb
    }


    rows = []


    for name, pred in predictions.items():

        mse = mean_squared_error(
            y_test,
            pred
        )


        rows.append({

            "Model": name,

            "MAE": mean_absolute_error(
                y_test,
                pred
            ),

            "MSE": mse,

            "RMSE": np.sqrt(mse),

            "R2 Score": r2_score(
                y_test,
                pred
            )
        })


    metrics_df = pd.DataFrame(rows)


    metrics_df = metrics_df.sort_values(
        by="R2 Score",
        ascending=False
    ).reset_index(drop=True)


    return (

        models,

        predictions,

        X_train,

        X_test,

        y_train,

        y_test,

        metrics_df
    )


def apply_style():

    st.markdown("""
    <style>

    .stApp {
        background: #f5f8fc;
        color: #26384a;
    }

    .stApp p,
    .stApp li,
    .stApp span,
    .stApp label {
        color: #26384a;
    }

    .stApp h1,
    .stApp h2,
    .stApp h3,
    .stApp h4,
    .stApp h5,
    .stApp h6 {
        color: #263238 !important;
    }

    .stMarkdown p,
    .stMarkdown li {
        color: #26384a !important;
    }

    [data-testid="stSidebar"] {
        background: #102a43;
    }

    [data-testid="stSidebar"] * {
        color: white !important;
    }

    [data-testid="stSidebar"] .stMarkdown h2 {
        color: #ffffff !important;
    }

    .main-title {

        background: linear-gradient(
            90deg,
            #2f6f9f,
            #3d91bd
        );

        padding: 22px 28px;

        border-radius: 14px;

        margin-bottom: 22px;
    }

    .main-title h1 {

        color: #263238 !important;

        margin: 0;

        font-size: 32px;

        font-weight: 700;
    }

    .main-title p {

        color: #ffffff !important;

        margin: 6px 0 0 0;

        font-size: 15px;
    }

    div[data-testid="stMetric"] {

        background: white;

        border-left: 5px solid #1976a8;

        padding: 12px;

        border-radius: 10px;

        box-shadow:
            0 2px 8px rgba(0,0,0,0.06);
    }

    div[data-testid="stMetric"] label,
    div[data-testid="stMetricLabel"],
    div[data-testid="stMetricLabel"] p {

        color: #52667a !important;
    }

    div[data-testid="stMetricValue"],
    div[data-testid="stMetricValue"] div {

        color: #123c69 !important;
    }

    div[data-testid="stMetricDelta"] {

        color: #1976a8 !important;
    }

    .section-title {

        color: #263238 !important;

        font-weight: 700;

        margin-top: 8px;
    }

    [data-testid="stWidgetLabel"] p,
    [data-testid="stWidgetLabel"] label,
    .stSelectbox label,
    .stNumberInput label,
    .stTextInput label,
    .stSlider label,
    .stRadio label,
    .stCheckbox label {

        color: #26384a !important;

        font-weight: 600;
    }

    .stTextInput input,
    .stNumberInput input,
    .stSelectbox input {

        color: #ffffff !important;

        -webkit-text-fill-color: #ffffff !important;
    }

    .stSelectbox [data-baseweb="select"] {

        background-color: #262733 !important;
    }

    .stSelectbox [data-baseweb="select"] * {

        color: #ffffff !important;

        -webkit-text-fill-color: #ffffff !important;
    }

    [data-baseweb="popover"] {

        background-color: white !important;
    }

    [data-baseweb="popover"] * {

        color: #26384a !important;
    }

    .stNumberInput div[data-baseweb="input"] {

        background-color: #262733 !important;
    }

    .stNumberInput div[data-baseweb="input"] input {

        color: #ffffff !important;

        -webkit-text-fill-color: #ffffff !important;
    }

    .stNumberInput button {

        color: #ffffff !important;

        background-color: #262733 !important;
    }

    .stTextInput div[data-baseweb="input"] {

        background-color: #262733 !important;
    }

    .stTextInput div[data-baseweb="input"] input {

        color: #ffffff !important;

        -webkit-text-fill-color: #ffffff !important;
    }

    .info-card {

        background: white;

        padding: 18px;

        border-radius: 12px;

        border: 1px solid #dbe7f2;

        margin-bottom: 15px;
    }

    .prediction-card {

        background: linear-gradient(
            135deg,
            #2f6f9f,
            #3d91bd
        );

        padding: 24px;

        border-radius: 14px;

        color: white;

        text-align: center;

        margin-top: 18px;
    }

    .prediction-card h2 {

        color: white !important;

        margin: 0;

        font-size: 34px;
    }

    .prediction-card p {

        color: white !important;
    }

    .small-note {

        color: #52667a !important;

        font-size: 13px;
    }

    [data-testid="stDataFrame"] {

        border-radius: 10px;
    }

    .stButton > button,
    .stFormSubmitButton > button {

        background: #1976a8;

        color: white !important;

        border: none;

        border-radius: 8px;

        font-weight: 600;
    }

    .stButton > button:hover,
    .stFormSubmitButton > button:hover {

        background: #123c69;

        color: white !important;
    }

    [data-testid="stAlert"] p {

        color: #123c69 !important;
    }

    </style>
    """, unsafe_allow_html=True)



def sidebar():

    st.sidebar.markdown(
        "## Student Dashboard"
    )


def get_project_data():

    raw = load_raw_data()

    df, encoders = prepare_data(raw)


    (
        models,
        predictions,
        X_train,
        X_test,
        y_train,
        y_test,
        metrics_df
    ) = train_all_models(
        df,
        encoders,
        "v3"
    )


    return (

        raw,

        df,

        encoders,

        models,

        predictions,

        X_train,

        X_test,

        y_train,

        y_test,

        metrics_df
    )


Overwriting utils.py


## Step 3: `main.py` (Dashboard home page)

In [13]:
%%writefile main.py
import streamlit as st
import matplotlib.pyplot as plt

from utils import apply_style, sidebar, get_project_data

st.set_page_config(
    page_title="Student Placement Dashboard",
    page_icon="",
    layout="wide"
)

apply_style()
sidebar()

try:
    raw, df, encoders, models, predictions, X_train, X_test, y_train, y_test, metrics_df = get_project_data()
except FileNotFoundError:
    st.error("studentplacement.csv nahi mili. CSV ko main.py ke same folder me rakho.")
    st.stop()

st.markdown("""
<div class="main-title">
    <h1> Student Placement & Salary Dashboard</h1>
</div>
""", unsafe_allow_html=True)

st.markdown('<h3 class="section-title">Project Overview</h3>', unsafe_allow_html=True)

placed = (raw["placement_status"].astype(str).str.strip().str.lower() == "placed").sum()
not_placed = len(raw) - placed
placement_rate = placed / len(raw) * 100
avg_package = raw["salary_package_lpa"].mean()

c1, c2, c3, c4 = st.columns(4)
c1.metric("Total Students", f"{len(raw):,}")
c2.metric("Placed Students", f"{placed:,}")
c3.metric("Placement Rate", f"{placement_rate:.2f}%")
c4.metric("Average Package", f"{avg_package:.2f} LPA")

st.markdown("---")

left, right = st.columns(2)

with left:
    st.subheader("Placement Status")
    counts = raw["placement_status"].value_counts()

    fig, ax = plt.subplots(figsize=(6, 4))
    ax.bar(counts.index, counts.values)
    ax.set_xlabel("Placement Status")
    ax.set_ylabel("Students")
    ax.set_title("Placed vs Not Placed")
    plt.xticks(rotation=15)
    st.pyplot(fig, use_container_width=True)

with right:
    st.subheader("Average Salary by Branch")
    branch_salary = raw.groupby("branch")["salary_package_lpa"].mean().sort_values(ascending=False)

    fig, ax = plt.subplots(figsize=(6, 4))
    branch_salary.plot(kind="bar", ax=ax)
    ax.set_xlabel("Branch")
    ax.set_ylabel("Average Package (LPA)")
    ax.set_title("Average Package by Branch")
    plt.xticks(rotation=40)
    st.pyplot(fig, use_container_width=True)

st.markdown("---")
st.subheader("Dataset Preview")
st.dataframe(raw.head(10), use_container_width=True)

st.caption(
    "Dataset: 100,000 rows and 26 columns. "
    "Target variable: salary_package_lpa."
)


Overwriting main.py


## Step 4: Dashboard all pages (`pages/` folder)

In [14]:
%%writefile pages/2_Data_Analysis.py
import streamlit as st
import matplotlib.pyplot as plt

from utils import apply_style, sidebar, get_project_data

st.set_page_config(
    page_title="Data Analysis",
    layout="wide"
)

apply_style()
sidebar()

try:
    raw, df, encoders, models, predictions, X_train, X_test, y_train, y_test, metrics_df = get_project_data()
except FileNotFoundError:
    st.error("Not Found .csv")
    st.stop()

st.markdown("""
<div class="main-title">
    <h1>Data Analysis</h1>
</div>
""", unsafe_allow_html=True)

c1, c2, c3, c4 = st.columns(4)
c1.metric("Rows", f"{len(raw):,}")
c2.metric("Columns", f"{len(raw.columns)}")
c3.metric("Missing Values", f"{int(raw.isnull().sum().sum())}")
c4.metric("Duplicate Rows", f"{int(raw.duplicated().sum())}")

st.markdown("---")

col1, col2 = st.columns(2)

with col1:
    st.subheader("CGPA Distribution")
    fig, ax = plt.subplots(figsize=(6, 4))
    ax.hist(raw["cgpa"], bins=20)
    ax.set_xlabel("CGPA")
    ax.set_ylabel("Students")
    ax.set_title("CGPA Distribution")
    st.pyplot(fig, use_container_width=True)

with col2:
    st.subheader("Salary Package Distribution")
    fig, ax = plt.subplots(figsize=(6, 4))
    ax.hist(raw["salary_package_lpa"], bins=20)
    ax.set_xlabel("Salary Package (LPA)")
    ax.set_ylabel("Students")
    ax.set_title("Salary Distribution")
    st.pyplot(fig, use_container_width=True)

st.markdown("---")

col1, col2 = st.columns(2)

with col1:
    st.subheader("CGPA vs Salary")
    fig, ax = plt.subplots(figsize=(6, 4))
    ax.scatter(
        raw["cgpa"],
        raw["salary_package_lpa"],
        alpha=0.20
    )
    ax.set_xlabel("CGPA")
    ax.set_ylabel("Salary Package (LPA)")
    ax.set_title("CGPA vs Salary")
    st.pyplot(fig, use_container_width=True)

with col2:
    st.subheader("Study Hours vs Salary")
    fig, ax = plt.subplots(figsize=(6, 4))
    ax.scatter(
        raw["study_hours_per_day"],
        raw["salary_package_lpa"],
        alpha=0.20
    )
    ax.set_xlabel("Study Hours Per Day")
    ax.set_ylabel("Salary Package (LPA)")
    ax.set_title("Study Hours vs Salary")
    st.pyplot(fig, use_container_width=True)

st.markdown("---")

st.subheader("Correlation with Salary")

numeric = raw.select_dtypes(include="number")
corr = numeric.corr()["salary_package_lpa"].sort_values(ascending=False)

st.dataframe(
    corr.to_frame("Correlation with Salary"),
    use_container_width=True
)

st.subheader("Numerical Dataset Summary")
st.dataframe(
    raw.describe().T.round(2),
    use_container_width=True
)


Overwriting pages/2_Data_Analysis.py


In [15]:
%%writefile pages/3_Model_Performance.py
import streamlit as st
import matplotlib.pyplot as plt

from utils import apply_style, sidebar, get_project_data

st.set_page_config(
    page_title="Model Performance",
    layout="wide"
)

apply_style()
sidebar()

try:
    raw, df, encoders, models, predictions, X_train, X_test, y_train, y_test, metrics_df = get_project_data()
except FileNotFoundError:
    st.error("Not Found .csv")
    st.stop()

st.markdown("""
<div class="main-title">
    <h1>Model Performance</h1>
</div>
""", unsafe_allow_html=True)

notebook_metrics = {
    "Linear Regression": {
        "MAE": 0.685,
        "MSE": 0.9204049403465818,
        "RMSE": 0.9593773711874707,
        "R2 Score": 0.9796589749531985,
        "Type": "Linear regression"
    },
    "Decision Tree": {
        "MAE": 0.632742,
        "MSE": 1.1541259,
        "RMSE": 1.074302517915694,
        "R2 Score": 0.9744937224802137,
        "Type": "Tree-based regression"
    },
    "Random Forest": {
        "MAE": 0.44850436499999996,
        "MSE": 0.5789051576945,
        "RMSE": 0.760858171865493,
        "R2 Score": 0.9872061482982128,
        "Type": "Ensemble regression"
    }
}

st.subheader("Complete Regression Model Details")

rows = []
for model_name, values in notebook_metrics.items():
    rows.append({
        "Model": model_name,
        "Type": values["Type"],
        "MAE": round(values["MAE"], 4),
        "MSE": round(values["MSE"], 4),
        "RMSE": round(values["RMSE"], 4),
        "R2 Score": round(values["R2 Score"], 4)
    })

details = st.dataframe(
    rows,
    use_container_width=True,
    hide_index=True
)

st.markdown("---")

c1, c2, c3 = st.columns(3)

for col, name in zip(
    [c1, c2, c3],
    ["Linear Regression", "Decision Tree", "Random Forest"]
):
    with col:
        st.markdown(f"### {name}")
        st.metric("R² Score", f"{notebook_metrics[name]['R2 Score']:.4f}")
        st.metric("MAE", f"{notebook_metrics[name]['MAE']:.4f}")
        st.metric("RMSE", f"{notebook_metrics[name]['RMSE']:.4f}")

st.markdown("---")

st.subheader("R² Score Comparison")

fig, ax = plt.subplots(figsize=(9, 4))
names = list(notebook_metrics.keys())
r2_values = [notebook_metrics[x]["R2 Score"] for x in names]

ax.bar(names, r2_values)
ax.set_ylabel("R² Score")
ax.set_ylim(0.90, 1.00)
ax.set_title("Regression Model R² Score")
for i, value in enumerate(r2_values):
    ax.text(i, value + 0.001, f"{value:.4f}", ha="center")
st.pyplot(fig, use_container_width=True)

st.markdown("---")

best_name = max(
    notebook_metrics,
    key=lambda x: notebook_metrics[x]["R2 Score"]
)

st.success(
    f"Best Model: {best_name} | "
    f"R² Score = {notebook_metrics[best_name]['R2 Score']:.4f}"
)


Overwriting pages/3_Model_Performance.py


In [16]:
%%writefile pages/4_Salary_Prediction.py
import streamlit as st
import pandas as pd
import os
import pickle

from utils import (
    apply_style,
    sidebar,
    get_project_data,
    SELECTED_FEATURES
)


st.set_page_config(
    page_title="Salary Prediction",
    page_icon="",
    layout="wide"
)


apply_style()
sidebar()


try:

    (
        raw,
        df,
        encoders,
        models,
        predictions,
        X_train,
        X_test,
        y_train,
        y_test,
        metrics_df
    ) = get_project_data()

except FileNotFoundError:

    st.error(
        "studentplacement.csv nahi mili. "
        "CSV ko project ke same folder me rakho."
    )

    st.stop()


model_path = os.path.join(
    os.path.dirname(os.path.dirname(__file__)),
    "saved_models",
    "random_forest.pkl"
)

with open(model_path, "rb") as file:
    rf = pickle.load(file)


st.markdown(
    """
    <div class="main-title">
        <h1>Salary Prediction</h1>
    </div>
    """,
    unsafe_allow_html=True
)


st.markdown(
    """
    <p style="
        color:#52667a;
        font-size:16px;
        margin-bottom:25px;
    ">
        Enter basic personal information and important
        academic & skill details to estimate salary package.
    </p>
    """,
    unsafe_allow_html=True
)


st.subheader("Personal Details")


with st.form("personal_details_form"):

    p1, p2, p3 = st.columns(3)

    with p1:

        student_id = st.number_input(
            "Student ID",
            min_value=int(raw["student_id"].min()),
            max_value=int(raw["student_id"].max()),
            value=int(raw["student_id"].median()),
            step=1
        )

    with p2:

        age = st.number_input(
            "Age",
            min_value=int(raw["age"].min()),
            max_value=int(raw["age"].max()),
            value=int(raw["age"].median()),
            step=1
        )

    with p3:

        gender = st.selectbox(
            "Gender",
            sorted(
                raw["gender"]
                .dropna()
                .astype(str)
                .unique()
            )
        )

    personal_submit = st.form_submit_button(
        "Save Personal Details"
    )


if personal_submit:

    st.session_state["personal_details"] = {
        "student_id": student_id,
        "age": age,
        "gender": gender
    }

    st.success("Personal details saved successfully.")


if "personal_details" in st.session_state:

    st.caption(
        "Personal details saved for this prediction session."
    )


st.markdown("---")


st.subheader(" Salary Prediction Details")


st.caption(
    "Ye fields salary prediction ke liye model mein use hongi."
)


with st.form("salary_prediction_form"):

    c1, c2, c3 = st.columns(3)

    with c1:

        cgpa = st.number_input(
            "CGPA",
            min_value=float(raw["cgpa"].min()),
            max_value=float(raw["cgpa"].max()),
            value=float(raw["cgpa"].median()),
            step=0.01
        )


        branch = st.selectbox(
            "Branch",
            sorted(
                raw["branch"]
                .dropna()
                .astype(str)
                .unique()
            )
        )


        college_tier = st.selectbox(
            "College Tier",
            sorted(
                raw["college_tier"]
                .dropna()
                .astype(str)
                .unique()
            )
        )


        internships_count = st.number_input(
            "Internships Count",
            min_value=int(
                raw["internships_count"].min()
            ),
            max_value=int(
                raw["internships_count"].max()
            ),
            value=int(
                raw["internships_count"].median()
            ),
            step=1
        )


    with c2:

        projects_count = st.number_input(
            "Projects Count",
            min_value=int(
                raw["projects_count"].min()
            ),
            max_value=int(
                raw["projects_count"].max()
            ),
            value=int(
                raw["projects_count"].median()
            ),
            step=1
        )


        certifications_count = st.number_input(
            "Certifications Count",
            min_value=int(
                raw["certifications_count"].min()
            ),
            max_value=int(
                raw["certifications_count"].max()
            ),
            value=int(
                raw["certifications_count"].median()
            ),
            step=1
        )


        coding_skill_score = st.number_input(
            "Coding Skill Score",
            min_value=float(
                raw["coding_skill_score"].min()
            ),
            max_value=float(
                raw["coding_skill_score"].max()
            ),
            value=float(
                raw["coding_skill_score"].median()
            ),
            step=0.01
        )


        aptitude_score = st.number_input(
            "Aptitude Score",
            min_value=float(
                raw["aptitude_score"].min()
            ),
            max_value=float(
                raw["aptitude_score"].max()
            ),
            value=float(
                raw["aptitude_score"].median()
            ),
            step=0.01
        )


    with c3:

        communication_skill_score = st.number_input(
            "Communication Skill Score",
            min_value=float(
                raw["communication_skill_score"].min()
            ),
            max_value=float(
                raw["communication_skill_score"].max()
            ),
            value=float(
                raw["communication_skill_score"].median()
            ),
            step=0.01
        )


        mock_interview_score = st.number_input(
            "Mock Interview Score",
            min_value=float(
                raw["mock_interview_score"].min()
            ),
            max_value=float(
                raw["mock_interview_score"].max()
            ),
            value=float(
                raw["mock_interview_score"].median()
            ),
            step=0.01
        )


        attendance_percentage = st.number_input(
            "Attendance Percentage",
            min_value=float(
                raw["attendance_percentage"].min()
            ),
            max_value=float(
                raw["attendance_percentage"].max()
            ),
            value=float(
                raw["attendance_percentage"].median()
            ),
            step=0.01
        )


        backlogs = st.number_input(
            "Backlogs",
            min_value=int(
                raw["backlogs"].min()
            ),
            max_value=int(
                raw["backlogs"].max()
            ),
            value=int(
                raw["backlogs"].median()
            ),
            step=1
        )


    st.markdown("---")


    predict_button = st.form_submit_button(
        " Predict Salary"
    )


if predict_button:

    st.session_state["personal_details"] = {
        "student_id": student_id,
        "age": age,
        "gender": gender
    }


    input_data = pd.DataFrame([{

        "cgpa": cgpa,

        "branch": branch,

        "college_tier": college_tier,

        "internships_count": internships_count,

        "projects_count": projects_count,

        "certifications_count": certifications_count,

        "coding_skill_score": coding_skill_score,

        "aptitude_score": aptitude_score,

        "communication_skill_score":
            communication_skill_score,

        "mock_interview_score":
            mock_interview_score,

        "attendance_percentage":
            attendance_percentage,

        "backlogs": backlogs

    }])


    for col, le in encoders.items():

        if col in input_data.columns:

            value = str(
                input_data.loc[0, col]
            )

            if value not in le.classes_:

                st.error(
                    f"{col} ke liye unknown category mili: {value}"
                )

                st.stop()

            input_data[col] = le.transform(
                [value]
            )[0]


    input_data = input_data[
        SELECTED_FEATURES
    ]


    predicted_salary = float(
        rf.predict(input_data)[0]
    )


    predicted_salary = max(
        0.0,
        predicted_salary
    )


    st.markdown("---")


    st.subheader(" Prediction Result")


    r1, r2 = st.columns(2)


    r1.metric(
        "Predicted Salary",
        f"{predicted_salary:.2f} LPA"
    )


    r2.metric(
        "Model",
        "Random Forest"
    )


    st.markdown(
        f"""
        <div class="prediction-card">
            <p>Estimated Salary Package</p>
            <h2>{predicted_salary:.2f} LPA</h2>
        </div>
        """,
        unsafe_allow_html=True
    )


    st.markdown("---")


    st.subheader("Personal Details")


    personal_display = pd.DataFrame(
        [st.session_state["personal_details"]]
    )


    st.dataframe(
        personal_display,
        use_container_width=True,
        hide_index=True
    )


    st.subheader("Salary Prediction Details")


    display_data = pd.DataFrame([{

        "CGPA": cgpa,

        "Branch": branch,

        "College Tier": college_tier,

        "Internships Count":
            internships_count,

        "Projects Count":
            projects_count,

        "Certifications Count":
            certifications_count,

        "Coding Skill Score":
            coding_skill_score,

        "Aptitude Score":
            aptitude_score,

        "Communication Skill Score":
            communication_skill_score,

        "Mock Interview Score":
            mock_interview_score,

        "Attendance Percentage":
            attendance_percentage,

        "Backlogs":
            backlogs

    }])


    st.dataframe(
        display_data,
        use_container_width=True,
        hide_index=True
    )


    st.info(
        "Ye prediction trained Random Forest model ka "
        "estimated result hai. Actual salary company, "
        "job role, skills aur placement process par "
        "depend kar sakti hai."
    )


Overwriting pages/4_Salary_Prediction.py


In [17]:
%%writefile pages/5_Bulk_Salary_Prediction.py
import streamlit as st
import pandas as pd
import os
import pickle

from utils import (
    apply_style,
    sidebar,
    SELECTED_FEATURES
)


st.set_page_config(
    page_title="Bulk Salary Prediction",
    page_icon="",
    layout="wide"
)

apply_style()
sidebar()


model_path = os.path.join(
    os.path.dirname(os.path.dirname(__file__)),
    "saved_models",
    "random_forest.pkl"
)

encoder_path = os.path.join(
    os.path.dirname(os.path.dirname(__file__)),
    "saved_models",
    "encoders.pkl"
)


try:

    with open(model_path, "rb") as file:
        rf = pickle.load(file)

    with open(encoder_path, "rb") as file:
        encoders = pickle.load(file)

except FileNotFoundError:

    st.error(
        "Saved model files nahi mili. "
        "saved_models folder check karo."
    )

    st.stop()


st.markdown(
    """
    <div class="main-title">
        <h1>Bulk Salary Prediction</h1>
    </div>
    """,
    unsafe_allow_html=True
)


st.markdown(
    """
    <p style="
        color:#52667a;
        font-size:16px;
        margin-bottom:25px;
    ">
        Upload multiple student details in CSV format
        and predict salary for all students at once.
    </p>
    """,
    unsafe_allow_html=True
)


st.subheader("Upload Student CSV")


uploaded_file = st.file_uploader(
    "Choose a CSV file",
    type=["csv"]
)

if uploaded_file is not None:

    data = pd.read_csv(uploaded_file)


    st.success(
        "CSV file uploaded successfully."
    )


    st.write(
        "Total Students:",
        len(data)
    )


    missing_columns = []

    for column in SELECTED_FEATURES:

        if column not in data.columns:

            missing_columns.append(column)


    if len(missing_columns) > 0:

        st.error(
            "CSV me required columns missing hain:"
        )

        for column in missing_columns:

            st.write(
                "-",
                column
            )

        st.stop()


    st.subheader("Uploaded Student Data")

    st.dataframe(
        data.head(100),
        use_container_width=True,
        hide_index=True
    )

    if len(data) > 100:

        st.caption(
            "Preview ke liye first 100 students dikhaye gaye hain."
        )



    st.markdown("---")


    predict_button = st.button(
        "Predict Salary for All Students"
    )


    if predict_button:

        # Make a copy
        input_data = data.copy()



        for col, le in encoders.items():

            if col in input_data.columns:

                # Convert values to string
                input_data[col] = (
                    input_data[col]
                    .astype(str)
                )


                encoding_map = dict(
                    zip(
                        le.classes_,
                        le.transform(
                            le.classes_
                        )
                    )
                )


                unknown_values = []

                for value in input_data[col].unique():

                    if value not in encoding_map:

                        unknown_values.append(value)


                if len(unknown_values) > 0:

                    st.error(
                        f"{col} me unknown category mili: "
                        f"{unknown_values}"
                    )

                    st.stop()


                # Apply encoding
                input_data[col] = (
                    input_data[col]
                    .map(encoding_map)
                )


        input_data = input_data[
            SELECTED_FEATURES
        ]



        with st.spinner(
            "Predicting salary for all students..."
        ):

            predictions = rf.predict(
                input_data
            )


        result_data = data.copy()


        result_data[
            "Predicted Salary (LPA)"
        ] = predictions


        result_data[
            "Predicted Salary (LPA)"
        ] = result_data[
            "Predicted Salary (LPA)"
        ].clip(
            lower=0
        )


        st.markdown("---")

        st.subheader(
            "Bulk Prediction Result"
        )


        st.dataframe(
            result_data,
            use_container_width=True,
            hide_index=True
        )


        st.markdown("---")

        st.subheader(
            "Prediction Summary"
        )


        col1, col2, col3 = st.columns(3)


        with col1:

            st.metric(
                "Total Students",
                len(result_data)
            )


        with col2:

            average_salary = (
                result_data[
                    "Predicted Salary (LPA)"
                ].mean()
            )

            st.metric(
                "Average Predicted Salary",
                f"{average_salary:.2f} LPA"
            )


        with col3:

            highest_salary = (
                result_data[
                    "Predicted Salary (LPA)"
                ].max()
            )

            st.metric(
                "Highest Predicted Salary",
                f"{highest_salary:.2f} LPA"
            )


        st.markdown("---")

        st.subheader(
            "Download Prediction Result"
        )


        csv_data = result_data.to_csv(
            index=False
        )


        st.download_button(
            label="Download Predicted CSV",
            data=csv_data,
            file_name="bulk_salary_predictions.csv",
            mime="text/csv"
        )


        st.success(
            "All student salaries predicted successfully."
        )


        st.info(
            "Ye predictions saved Random Forest model "
            "ke basis par generate ki gayi hain. "
            "Actual salary company, job role, skills "
            "aur placement process par depend kar sakti hai."
        )


Overwriting pages/5_Bulk_Salary_Prediction.py


## Step 5: Models train 

In [18]:
import logging, warnings
logging.getLogger("streamlit").setLevel(logging.ERROR)   
warnings.filterwarnings("ignore")

import utils
raw, df, encoders, models, predictions, X_train, X_test, y_train, y_test, metrics_df = utils.get_project_data()
print("Rows:", len(raw), "| Placed students (salary models ka data):", len(X_train) + len(X_test))
metrics_df

Rows: 100000 | Placed students (salary models ka data): 54459


,Model,MAE,MSE,RMSE,R2 Score
0,Linear Regression,0.798317,0.997844,0.998921,0.606107
1,Gradient Boosting,0.801410,1.005620,1.002806,0.603037
2,Random Forest,0.819785,1.048948,1.024181,0.585934
3,Extra Trees,0.826768,1.067166,1.033037,0.578742
4,Decision Tree,0.980442,1.510328,1.228954,0.403806


In [19]:
for f in sorted(os.listdir("saved_models")):
    print(f"{f:25s} {os.path.getsize(os.path.join('saved_models', f)) / 1e6:8.1f} MB")

.ipynb_checkpoints             0.0 MB
README.txt                     0.0 MB
decision_tree.pkl              0.7 MB
encoders.pkl                   0.0 MB
extra_trees.pkl             1677.6 MB
gradient_boosting.pkl          0.7 MB
linear_regression.pkl          0.0 MB
random_forest.pkl            852.7 MB


## Step 6: Dashboard run

In [20]:
import time, webbrowser
from IPython.display import IFrame, display

PORT = 8501
dash = subprocess.Popen(
    [sys.executable, "-m", "streamlit", "run", "main.py",
     "--server.port", str(PORT), "--server.headless", "true",
     "--browser.gatherUsageStats", "false"],
    cwd=os.getcwd()
)
time.sleep(6)
print(f"Dashboard chal raha hai: http://localhost:{PORT}")
try:
    webbrowser.open(f"http://localhost:{PORT}")
except Exception:
    pass
display(IFrame(f"http://localhost:{PORT}", width="100%", height=700))

Dashboard chal raha hai: http://localhost:8501


## Step 7: Dashboard band karo

In [21]:
dash.terminate()
print("Dashboard band ho gaya.")

Dashboard band ho gaya.
